# DLM em painel — base corrigida

Efeitos fixos de UF e mês-ano, Δlog(I) e Δlog(1+D). O exercício 0..K seleciona K por AIC entre 1 e 24; a sensibilidade 3..K seleciona entre 3 e 24. Cada exercício compara candidatos sobre amostra comum. Covariâncias e ajustes seguem as especificações documentadas no código.

In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'outputs').exists(): ROOT=ROOT.parent
B=ROOT/'outputs/tables/modalidades_excluidas_20261104'
import json
manifest=json.loads((B/'manifesto.json').read_text(encoding='utf-8'))
print('Base PF com três exclusões:', manifest['base_sha256'])
print('Período:', manifest['periodo'], 'Observações:', manifest['linhas'])


Base PF com três exclusões: d795e0ed176cc4628e1b02a81bc32ecab449f24aefc3e9be91d291479f46c043
Período: ['2013-01', '2024-12'] Observações: 3888


In [2]:
s=pd.read_csv(B/'dlm_aic_bic/dlm_selecao.csv'); print(s.groupby(['criterio','status']).size().to_string()); print(s[['periodo','tipologia','criterio','K','status']].to_string(index=False))

criterio  status              
AIC       estimado                29
          suporte insuficiente     3
BIC       estimado                29
          suporte insuficiente     3
periodo                            tipologia criterio    K               status
    Pré                     tipo_alagamentos      AIC  1.0             estimado
    Pré                     tipo_alagamentos      BIC  1.0             estimado
    Pré                 tipo_chuvas_intensas      AIC  6.0             estimado
    Pré                 tipo_chuvas_intensas      BIC  1.0             estimado
    Pré             tipo_doencas_infecciosas      AIC  1.0             estimado
    Pré             tipo_doencas_infecciosas      BIC  1.0             estimado
    Pré                      tipo_enxurradas      AIC  1.0             estimado
    Pré                      tipo_enxurradas      BIC  1.0             estimado
    Pré                          tipo_erosao      AIC  1.0             estimado
    Pré              

In [3]:
t=pd.read_csv(B/'dlm_top2/resumo_inferencia_top2.csv'); print('Contrastes por posição AIC e banda DK:'); print(t.to_string(index=False)); print('As posições 1 e 2 são especificações alternativas, não efeitos independentes.')

Contrastes por posição AIC e banda DK:
 posicao_AIC banda_DK  hipoteses  nominal  com_ajustes
           1     dk12        145       58            9
           1      dk6        145       56            5
           1 dk_regra        145       54            5
           2     dk12        145       55            4
           2      dk6        145       46            2
           2 dk_regra        145       46            1
As posições 1 e 2 são especificações alternativas, não efeitos independentes.


In [4]:
r=pd.read_csv(B/'dlm0_inferencia/resultados.csv'); print('Exercício ampliado: total climático, grupos e tipologias'); print(r.groupby('covariancia').agg(nominais=('p',lambda x:x.lt(.05).sum()),ajustadas=('p_Holm_familia',lambda x:x.lt(.05).sum())).to_string()); print('Total climático central:'); print(r[r.nivel.eq('Central') & r.covariancia.eq('DK_regra')][['periodo','endpoint','p','p_Holm_familia']].to_string(index=False))

Exercício ampliado: total climático, grupos e tipologias
             nominais  ajustadas
covariancia                     
CR1                54          4
DK12               71         10
DK6                69          5
DK_regra           66          5
Total climático central:
periodo   endpoint        p  p_Holm_familia
    Pré      beta0 0.552019        1.000000
    Pré     soma0K 0.805714        1.000000
    Pré     soma1K 0.999350        1.000000
    Pré conjunto0K 0.552019        1.000000
    Pré conjunto1K 0.999350        1.000000
  Total      beta0 0.050987        1.000000
  Total     soma0K 0.041658        0.999795
  Total     soma1K 0.078120        1.000000
  Total conjunto0K 0.050987        1.000000
  Total conjunto1K 0.078120        1.000000


In [5]:
print((B/'dlm3/resumo.json').read_text(encoding='utf-8'))

{
  "base_sha256": "d795e0ed176cc4628e1b02a81bc32ecab449f24aefc3e9be91d291479f46c043",
  "periodos": [
    "Pré",
    "Total"
  ],
  "K_candidatos": [
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24
  ],
  "lags_incluidos": "somente [3,..,K] (sem contemporâneo, lag1, lag2)",
  "resultado_historico_preservado": true,
  "reconciliacao_total_climatico": 37643,
  "tipologias_climaticas": [
    "tipo_alagamentos",
    "tipo_chuvas_intensas",
    "tipo_enxurradas",
    "tipo_estiagem_e_seca",
    "tipo_granizo",
    "tipo_incendio_florestal",
    "tipo_inundacoes",
    "tipo_onda_de_calor_e_baixa_umidade",
    "tipo_onda_de_frio",
    "tipo_tornado",
    "tipo_vendavais_e_ciclones"
  ],
  "n_aplicacoes": 42,
  "estimaveis": 39,
  "K_selecionados": {
    "3.0": 26,
    "5.0": 3,
    "14.0": 3,
    "4.0": 3,
    "9.0": 2,
    "6.0": 1,
    "16.0": 1
  },
  "rejeicoes_por_band

Os resultados com 3..K mantêm a sensibilidade a 90 dias. O indicador do SCR é estoque de saldos; esse corte não determina o instante de início de um efeito. Correções da grade e de famílias não equivalem a inferência seletiva exata.